<a href="https://colab.research.google.com/github/lricci03/Hands-on-ML/blob/main/c10/c10_ex15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 10 - exercise 15. Build and train a classification MLP on the CoverType dataset.

## Load the data

In [ ]:
# load the dataset
from sklearn.datasets import fetch_covtype

# Replace fetch_openml with the optimized scikit-learn loader
covtype = fetch_covtype(as_frame=True)

# Your existing logic remains exactly the same
X = covtype.data
y = covtype.target

Looking at the data e.g. how many classes?

In [ ]:
print(X.shape)
print(y.shape)
print(y.value_counts())
print(y.min(), y.max())

(581012, 54)
(581012,)
Cover_Type
2    283301
1    211840
3     35754
7     20510
6     17367
5      9493
4      2747
Name: count, dtype: int64
1 7


## Split into train, validation and test set

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    covtype.data, covtype.target, random_state=42)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size = 0.2, random_state=42
)

In [ ]:
import torch

Convert the data to tensors.

The data are pandas DataFrames.

In [ ]:
X_train.shape

(348607, 54)

In [ ]:
X_train = torch.FloatTensor(X_train.values)
X_valid = torch.FloatTensor(X_valid.values)
X_test = torch.FloatTensor(X_test.values)

y_train = torch.FloatTensor(y_train.values)
y_valid = torch.FloatTensor(y_valid.values)
y_test = torch.FloatTensor(y_test.values)

In [ ]:
y_train.shape

torch.Size([348607])

The targets range from 1 to 7, but the nn.CrossEntropyLoss expects it to start at 0, which is why we subtract 1 from the targets.

.long converts the targets to 64-bit integers

In [ ]:
y_train = (y_train - 1).long()
y_valid = (y_valid - 1).long()
y_test = (y_test - 1).long()

## Standardize the data

In [ ]:
means = X_train.mean(dim=0,keepdims=True) # dim=0 to take the mean along columns
stds = X_train.std(dim=0,keepdims=True)
X_train_std = (X_train - means)/stds
X_valid_std = (X_valid - means)/stds
X_test_std = (X_test - means)/stds

In [ ]:
'''y_train = y_train.reshape(-1,1)
y_valid = y_valid.reshape(-1,1)
y_test = y_test.reshape(-1,1)'''

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
train_std_dataset = TensorDataset(X_train_std,y_train)
valid_std_dataset = TensorDataset(X_valid_std, y_valid)
test_std_dataset = TensorDataset(X_test_std, y_test)

## Create data loaders

In [ ]:
train_std_loader = DataLoader(train_std_dataset, batch_size = 32, shuffle=True)
valid_std_loader = DataLoader(valid_std_dataset, batch_size = 32)
test_std_loader = DataLoader(test_std_dataset, batch_size = 32)

## Custom MLP module for classification

From Chapter 9 - ex10: we used three layers with 200, 100 and 50 neurons, respectively.

In [ ]:
import torch.nn as nn

In [ ]:
if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [ ]:
class MLP_clf(nn.Module):
  def __init__(self,n_inputs,n_classes):
    super().__init__()
    self.stack = nn.Sequential(
        nn.Linear(n_inputs,200),
        nn.ReLU(),
        nn.Linear(200,100),
        nn.ReLU(),
        nn.Linear(100,50),
        nn.ReLU(),
        nn.Linear(50,n_classes)
    )

  def forward(self,X):
    return self.stack(X)

## Define a train function using minibatches

In [ ]:
!pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.8 MB/s eta 0:00:00


In [ ]:
import torchmetrics

In [ ]:
def train(model, optimizer, criterion, train_loader, n_epochs):
  train_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
  for epoch in range(n_epochs):
    model.train()
    train_accuracy.reset()

    for X_train_batch, y_train_batch in train_loader:
      X_train_batch, y_train_batch = X_train_batch.to(device), y_train_batch.to(device)
      y_pred = model(X_train_batch)
      y_pred_class = y_pred.argmax(dim=1) # unnecessary, with torchmetrics.Accuracy we can directly use y_pred
      loss = criterion(y_pred, y_train_batch)
      train_accuracy.update(y_pred_class,y_train_batch)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    epoch_accuracy = train_accuracy.compute()
    print(f'Epoch {epoch + 1}/{n_epochs}, Accuracy: {epoch_accuracy:.4f}')


In [ ]:
n_inputs = len(covtype.feature_names)  # == 54
n_classes = len(set(covtype.target))  # == 7

In [ ]:
model = MLP_clf(n_inputs=54, n_classes=7)
model = model.to(device)

In [ ]:
learning_rate = 0.001
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
xentropy = nn.CrossEntropyLoss()

Training with a reduced train set

In [ ]:
train_data_reduced = TensorDataset(X_train_std[:100000],y_train[:100000])
train_loader_reduced = DataLoader(train_data_reduced, batch_size=32, shuffle=True)

In [ ]:
train(model,optimizer,xentropy,train_loader_reduced,20)

Epoch 1/20, Accuracy: 0.4642
Epoch 2/20, Accuracy: 0.4877
Epoch 3/20, Accuracy: 0.5169
Epoch 4/20, Accuracy: 0.6306
Epoch 5/20, Accuracy: 0.6540
Epoch 6/20, Accuracy: 0.6795
Epoch 7/20, Accuracy: 0.6985
Epoch 8/20, Accuracy: 0.7099
Epoch 9/20, Accuracy: 0.7164
Epoch 10/20, Accuracy: 0.7220
Epoch 11/20, Accuracy: 0.7272
Epoch 12/20, Accuracy: 0.7296
Epoch 13/20, Accuracy: 0.7324
Epoch 14/20, Accuracy: 0.7338
Epoch 15/20, Accuracy: 0.7351
Epoch 16/20, Accuracy: 0.7359
Epoch 17/20, Accuracy: 0.7367
Epoch 18/20, Accuracy: 0.7377
Epoch 19/20, Accuracy: 0.7400
Epoch 20/20, Accuracy: 0.7409


In 20 epochs, with train set reduced to 100'000 data points and learning rate of 0.001 we reach accuracy of 0.7409

## Train function with accuracy on train and validation sets

In [ ]:
def train2(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
  train_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
  valid_accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=7).to(device)
  for epoch in range(n_epochs):
    model.train()
    train_accuracy.reset()
    total_loss = 0.

    for X_train_batch, y_train_batch in train_loader:
      X_train_batch, y_train_batch = X_train_batch.to(device), y_train_batch.to(device)
      y_pred = model(X_train_batch)
      y_pred_class = y_pred.argmax(dim=1) # unnecessary, with torchmetrics.Accuracy we can directly use y_pred
      loss = criterion(y_pred, y_train_batch)
      total_loss += loss.item()
      train_accuracy.update(y_pred_class,y_train_batch)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    epoch_accuracy = train_accuracy.compute()
    mean_loss = total_loss/len(train_loader)
    # print(f'Epoch {epoch + 1}/{n_epochs}: Train Loss: {mean_loss:.4f} Train Accuracy: {epoch_accuracy:.4f}')

    with torch.no_grad():
      model.eval()
      valid_accuracy.reset()

      for X_valid_batch, y_valid_batch in valid_loader:
        X_valid_batch, y_valid_batch = X_valid_batch.to(device), y_valid_batch.to(device)
        y_valid_pred = model(X_valid_batch)
        y_valid_class = y_valid_pred.argmax(dim=1)
        valid_accuracy.update(y_valid_class, y_valid_batch)
      epoch_valid_accuracy = valid_accuracy.compute()

    print(f'Epoch {epoch + 1}/{n_epochs}: Train Loss: {mean_loss:.4f} Train Accuracy: {epoch_accuracy:.4f}, Validation Accuracy: {epoch_valid_accuracy:.4f}')

In [ ]:
model2 = MLP_clf(n_inputs=54, n_classes=7)
model2 = model2.to(device)

In [ ]:
learning_rate = 0.001
optimizer2 = torch.optim.SGD(model2.parameters(), lr=learning_rate)

In [ ]:
xentropy = nn.CrossEntropyLoss()

In [ ]:
train2(model2,optimizer2,xentropy,train_loader_reduced,valid_std_loader,10)

Epoch 1/10: Train Loss: 1.5321 Train Accuracy: 0.4331, Validation Accuracy: 0.4888
Epoch 2/10: Train Loss: 1.1507 Train Accuracy: 0.4892, Validation Accuracy: 0.4970
Epoch 3/10: Train Loss: 1.0022 Train Accuracy: 0.5545, Validation Accuracy: 0.6293
Epoch 4/10: Train Loss: 0.8893 Train Accuracy: 0.6491, Validation Accuracy: 0.6558
Epoch 5/10: Train Loss: 0.8108 Train Accuracy: 0.6657, Validation Accuracy: 0.6766
Epoch 6/10: Train Loss: 0.7638 Train Accuracy: 0.6843, Validation Accuracy: 0.6911
Epoch 7/10: Train Loss: 0.7314 Train Accuracy: 0.6985, Validation Accuracy: 0.7036
Epoch 8/10: Train Loss: 0.7092 Train Accuracy: 0.7082, Validation Accuracy: 0.7124
Epoch 9/10: Train Loss: 0.6944 Train Accuracy: 0.7142, Validation Accuracy: 0.7194
Epoch 10/10: Train Loss: 0.6838 Train Accuracy: 0.7210, Validation Accuracy: 0.7239


# Chapter 10 - ex 15 (d): reach 93% accuracy. Perform hyperparameter search.

Train with different learning rates.

In [ ]:
torch.manual_seed(42)
model3 = MLP_clf(n_inputs=54, n_classes=7)
model3 = model3.to(device)
xentropy = nn.CrossEntropyLoss()

for lr in [0.2, 0.1, 0.05, 0.001]:
  # learning_rate = lr
  optimizer3 = torch.optim.SGD(model3.parameters(), lr=lr)
  train2(model3,optimizer3,xentropy,train_loader_reduced,valid_std_loader,10)

Epoch 1/10: Train Loss: 0.6460 Train Accuracy: 0.7239, Validation Accuracy: 0.7573
Epoch 2/10: Train Loss: 0.5375 Train Accuracy: 0.7681, Validation Accuracy: 0.7854
Epoch 3/10: Train Loss: 0.4881 Train Accuracy: 0.7922, Validation Accuracy: 0.7968
Epoch 4/10: Train Loss: 0.4546 Train Accuracy: 0.8071, Validation Accuracy: 0.8103
Epoch 5/10: Train Loss: 0.4279 Train Accuracy: 0.8194, Validation Accuracy: 0.8166
Epoch 6/10: Train Loss: 0.4079 Train Accuracy: 0.8298, Validation Accuracy: 0.8153
Epoch 7/10: Train Loss: 0.3917 Train Accuracy: 0.8359, Validation Accuracy: 0.8353
Epoch 8/10: Train Loss: 0.3796 Train Accuracy: 0.8424, Validation Accuracy: 0.8281
Epoch 9/10: Train Loss: 0.3663 Train Accuracy: 0.8485, Validation Accuracy: 0.8427
Epoch 10/10: Train Loss: 0.3565 Train Accuracy: 0.8518, Validation Accuracy: 0.8504
Epoch 1/10: Train Loss: 0.2919 Train Accuracy: 0.8795, Validation Accuracy: 0.8770
Epoch 2/10: Train Loss: 0.2794 Train Accuracy: 0.8834, Validation Accuracy: 0.8770
Epo

The differences in accuracy from the start are given by


*   Different random weights assigned when we initialize the model: use `torch.manual_seed(42)` before initializing every model.
*   Different learning rates: already after the first epoch the weights can be widly different if the learning rate is high
* shuffling: in the train data loader we set `shuffle=True`, hence each run sees the data in a different order, which means the weights are updated differently.

We reached 0.9171 on the validation set with decreasing learning rates with 10 epochs each, for a total of 40 epochs.
What happens if we do 40 epochs at the highest learning rate?

In [ ]:
torch.manual_seed(42)
model4 = MLP_clf(n_inputs=54, n_classes=7)
model4 = model4.to(device)
xentropy = nn.CrossEntropyLoss()

learning_rate = 0.2
optimizer4 = torch.optim.SGD(model4.parameters(), lr=learning_rate)
train2(model4,optimizer4,xentropy,train_loader_reduced,valid_std_loader,40)

Epoch 1/40: Train Loss: 0.6460 Train Accuracy: 0.7239, Validation Accuracy: 0.7573
Epoch 2/40: Train Loss: 0.5375 Train Accuracy: 0.7681, Validation Accuracy: 0.7854
Epoch 3/40: Train Loss: 0.4881 Train Accuracy: 0.7922, Validation Accuracy: 0.7968
Epoch 4/40: Train Loss: 0.4546 Train Accuracy: 0.8071, Validation Accuracy: 0.8103
Epoch 5/40: Train Loss: 0.4279 Train Accuracy: 0.8194, Validation Accuracy: 0.8166
Epoch 6/40: Train Loss: 0.4079 Train Accuracy: 0.8298, Validation Accuracy: 0.8153
Epoch 7/40: Train Loss: 0.3917 Train Accuracy: 0.8359, Validation Accuracy: 0.8353
Epoch 8/40: Train Loss: 0.3796 Train Accuracy: 0.8424, Validation Accuracy: 0.8281
Epoch 9/40: Train Loss: 0.3663 Train Accuracy: 0.8485, Validation Accuracy: 0.8427
Epoch 10/40: Train Loss: 0.3565 Train Accuracy: 0.8518, Validation Accuracy: 0.8504
Epoch 11/40: Train Loss: 0.3467 Train Accuracy: 0.8573, Validation Accuracy: 0.8560
Epoch 12/40: Train Loss: 0.3413 Train Accuracy: 0.8589, Validation Accuracy: 0.8484
E

Let's use the full training set.

In [ ]:
torch.manual_seed(42)
model5 = MLP_clf(n_inputs=54, n_classes=7)
model5 = model5.to(device)
xentropy = nn.CrossEntropyLoss()

for lr in [0.2, 0.1, 0.05, 0.001]:
  # learning_rate = lr
  optimizer5 = torch.optim.SGD(model5.parameters(), lr=lr)
  train2(model5,optimizer5,xentropy,train_std_loader,valid_std_loader,10)

Epoch 1/10: Train Loss: 0.5466 Train Accuracy: 0.7661, Validation Accuracy: 0.8061
Epoch 2/10: Train Loss: 0.4254 Train Accuracy: 0.8210, Validation Accuracy: 0.8319
Epoch 3/10: Train Loss: 0.3781 Train Accuracy: 0.8425, Validation Accuracy: 0.8481
Epoch 4/10: Train Loss: 0.3526 Train Accuracy: 0.8550, Validation Accuracy: 0.8592
Epoch 5/10: Train Loss: 0.3615 Train Accuracy: 0.8536, Validation Accuracy: 0.8619
Epoch 6/10: Train Loss: 0.3535 Train Accuracy: 0.8602, Validation Accuracy: 0.8613
Epoch 7/10: Train Loss: 0.3325 Train Accuracy: 0.8664, Validation Accuracy: 0.8683
Epoch 8/10: Train Loss: 0.3143 Train Accuracy: 0.8737, Validation Accuracy: 0.8709
Epoch 9/10: Train Loss: 0.3043 Train Accuracy: 0.8772, Validation Accuracy: 0.8751
Epoch 10/10: Train Loss: 0.2966 Train Accuracy: 0.8804, Validation Accuracy: 0.8800
Epoch 1/10: Train Loss: 0.2309 Train Accuracy: 0.9064, Validation Accuracy: 0.9033
Epoch 2/10: Train Loss: 0.2206 Train Accuracy: 0.9108, Validation Accuracy: 0.9057
Epo

Collapsing of the model at learning rate = 0.005, Epoch 7/10. The learning rate was likely too high and we exited the minimum region.

The following training algorithm saves the model with best accuracy.

In [ ]:
import copy

def train3(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
  train_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
  valid_accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=7).to(device)

  best_acc = 0.0
  best_state = None

  for epoch in range(n_epochs):
    model.train()
    train_accuracy.reset()
    total_loss = 0.

    for X_train_batch, y_train_batch in train_loader:
      X_train_batch, y_train_batch = X_train_batch.to(device), y_train_batch.to(device)
      y_pred = model(X_train_batch)
      y_pred_class = y_pred.argmax(dim=1)
      loss = criterion(y_pred, y_train_batch)
      total_loss += loss.item()
      train_accuracy.update(y_pred_class,y_train_batch)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    epoch_accuracy = train_accuracy.compute()
    mean_loss = total_loss/len(train_loader)

    with torch.no_grad():
      model.eval()
      valid_accuracy.reset()

      for X_valid_batch, y_valid_batch in valid_loader:
        X_valid_batch, y_valid_batch = X_valid_batch.to(device), y_valid_batch.to(device)
        y_valid_pred = model(X_valid_batch)
        y_valid_class = y_valid_pred.argmax(dim=1)
        valid_accuracy.update(y_valid_class, y_valid_batch)
      epoch_valid_accuracy = valid_accuracy.compute()

      if epoch_valid_accuracy > best_acc:
        best_acc = epoch_valid_accuracy
        best_state = copy.deepcopy(model.state_dict())


    print(f'Epoch {epoch + 1}/{n_epochs}: Train Loss: {mean_loss:.4f} Train Accuracy: {epoch_accuracy:.4f}, Validation Accuracy: {epoch_valid_accuracy:.4f}')
  if best_state is not None:
    model.load_state_dict(best_state)

  return model, best_acc

In [ ]:
torch.manual_seed(42)
model6 = MLP_clf(n_inputs=54, n_classes=7)
model6 = model6.to(device)
xentropy = nn.CrossEntropyLoss()

for lr in [0.2, 0.1, 0.05, 0.001]:
  # learning_rate = lr
  optimizer6 = torch.optim.SGD(model6.parameters(), lr=lr)
  model, best_acc = train3(model6, optimizer6, xentropy, train_std_loader, valid_std_loader, 10)

print(f"Best validation accuracy: {best_acc:.4f}")

Epoch 1/10: Train Loss: 0.5466 Train Accuracy: 0.7661, Validation Accuracy: 0.8061
Epoch 2/10: Train Loss: 0.4254 Train Accuracy: 0.8210, Validation Accuracy: 0.8319
Epoch 3/10: Train Loss: 0.3781 Train Accuracy: 0.8425, Validation Accuracy: 0.8481
Epoch 4/10: Train Loss: 0.3526 Train Accuracy: 0.8550, Validation Accuracy: 0.8592
Epoch 5/10: Train Loss: 0.3615 Train Accuracy: 0.8536, Validation Accuracy: 0.8619
Epoch 6/10: Train Loss: 0.3535 Train Accuracy: 0.8602, Validation Accuracy: 0.8613
Epoch 7/10: Train Loss: 0.3325 Train Accuracy: 0.8664, Validation Accuracy: 0.8683
Epoch 8/10: Train Loss: 0.3143 Train Accuracy: 0.8737, Validation Accuracy: 0.8709
Epoch 9/10: Train Loss: 0.3043 Train Accuracy: 0.8772, Validation Accuracy: 0.8751
Epoch 10/10: Train Loss: 0.2966 Train Accuracy: 0.8804, Validation Accuracy: 0.8800
Epoch 1/10: Train Loss: 0.2309 Train Accuracy: 0.9064, Validation Accuracy: 0.9033
Epoch 2/10: Train Loss: 0.2206 Train Accuracy: 0.9108, Validation Accuracy: 0.9057
Epo

## Fine-Tuning NN Hyperparameters with Optuna

### Set up train() function for Optuna

Optuna needs an `objective` function whose inputs are the trial objects and whose output is a scalar value representing the model performance.

We need to change our `train2()` function to return the validation accuracy (this is the model performance metric that we want to use).

In [ ]:
def train3(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
  train_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
  valid_accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=7).to(device)
  for epoch in range(n_epochs):
    model.train()
    train_accuracy.reset()
    total_loss = 0.

    for X_train_batch, y_train_batch in train_loader:
      X_train_batch, y_train_batch = X_train_batch.to(device), y_train_batch.to(device)
      y_pred = model(X_train_batch)
      y_pred_class = y_pred.argmax(dim=1) # unnecessary, with torchmetrics.Accuracy we can directly use y_pred
      loss = criterion(y_pred, y_train_batch)
      total_loss += loss.item()
      train_accuracy.update(y_pred_class,y_train_batch)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    epoch_accuracy = train_accuracy.compute()
    mean_loss = total_loss/len(train_loader)

    with torch.no_grad():
      model.eval()
      valid_accuracy.reset()

      for X_valid_batch, y_valid_batch in valid_loader:
        X_valid_batch, y_valid_batch = X_valid_batch.to(device), y_valid_batch.to(device)
        y_valid_pred = model(X_valid_batch)
        y_valid_class = y_valid_pred.argmax(dim=1)
        valid_accuracy.update(y_valid_class, y_valid_batch)
      epoch_valid_accuracy = valid_accuracy.compute()

  return epoch_valid_accuracy

In [ ]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 29.6 MB/s eta 0:00:00


In [ ]:
import optuna

### We define the objective function

In [ ]:
def objective(trial):
  learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
  model = MLP_clf(n_inputs = 54, n_classes = 7)
  model = model.to(device)
  optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
  xentropy = nn.CrossEntropyLoss()
  validation_accuracy = train3(model, optimizer, xentropy, train_std_loader, valid_std_loader, 10)
  return validation_accuracy

In [ ]:
torch.manual_seed(42) # to ensure reproducibility
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='maximize',sampler=sampler) # by default optuna minimizes but we want to maximize the objective accuracy
study.optimize(objective, n_trials=4)


[I 2026-08-03 12:13:35,617] A new study created in memory with name: no-name-1994318d-c640-4910-97aa-c6424a41b75a
[I 2026-08-03 12:18:42,504] Trial 0 finished with value: 0.7321920394897461 and parameters: {'learning_rate': 0.00031489116479568613}. Best is trial 0 with value: 0.7321920394897461.
[I 2026-08-03 12:23:34,700] Trial 1 finished with value: 0.8904098868370056 and parameters: {'learning_rate': 0.06351221010640701}. Best is trial 1 with value: 0.8904098868370056.
[I 2026-08-03 12:28:24,050] Trial 2 finished with value: 0.8454309701919556 and parameters: {'learning_rate': 0.008471801418819975}. Best is trial 1 with value: 0.8904098868370056.
[I 2026-08-03 12:33:12,847] Trial 3 finished with value: 0.7976179718971252 and parameters: {'learning_rate': 0.0024810409748678114}. Best is trial 1 with value: 0.8904098868370056.


In [ ]:
study.best_params

{'learning_rate': 0.06351221010640701}

In [ ]:
study.best_value

0.8904098868370056

Training with the obtained learning rate.

In [ ]:
lr = study.best_params['learning_rate']

In [ ]:
torch.manual_seed(42)
model7 = MLP_clf(n_inputs=54, n_classes=7)
model7 = model7.to(device)
xentropy = nn.CrossEntropyLoss()

optimizer7 = torch.optim.SGD(model7.parameters(), lr=lr)
valid_accuracy = train3(model7, optimizer7, xentropy, train_std_loader, valid_std_loader, 40)

In [ ]:
valid_accuracy

tensor(0.9224, device='cuda:0')

Accuracy on the test set

In [ ]:
def evaluate_fct(model, data_loader):
  model.eval()
  accuracy = torchmetrics.Accuracy(task='multiclass',num_classes=7).to(device)
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch =X_batch.to(device), y_batch.to(device)
      test_pred_logits = model(X_batch)
      test_pred = test_pred_logits.argmax(dim=1) # index of the largest logit
      accuracy.update(test_pred, y_batch)
    test_accuracy = accuracy.compute()
  print(f'test accuracy: {test_accuracy}')


Accuracy of model7:

In [ ]:
evaluate_fct(model7, test_std_loader)

test accuracy: 0.9214818477630615


Accuracy of model6:

In [ ]:
evaluate_fct(model6, test_std_loader)

test accuracy: 0.9427481889724731
